# Подбор параметров модели по F1

Сравниваем настройки логистической регрессии L2 на текущей таблице `features.csv`. Подбираем силу регуляризации `C`, множитель весов дополнительных меток и порог вероятности сделки. Для сравнения используем только отложенные объявления из `verified.csv`. Запускайте из корня проекта или из папки ноутбука.

## 1. Настроить эксперимент и загрузить данные

In [ ]:
import os
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name in {"analysis", "feature", "models"}:
    repo_root = repo_root.parent

assert (repo_root / "requirements.txt").is_file() and (repo_root / "data/listings.csv").is_file(), (
    "Запустите ноутбук из корня репозитория или из папки, в которой он лежит"
)
os.chdir(repo_root)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [ ]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
EXTENDED_VERIFIED_PATH = Path("analysis/extend_verified.csv")
LISTINGS_PATH = Path("data/listings.csv")

C_VALUES = [0.1, 0.3, 1.0, 3.0, 10.0]
EXTRA_WEIGHT_SCALES = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
INCLUDE_WEIGHT_08 = True
THRESHOLDS = np.round(np.arange(0.20, 0.71, 0.01), 2)
CV_SEEDS = [11, 23, 42]
INNER_SEED = 42

In [ ]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.is_unique, "Повторяются ID в проверенной разметке"
assert verified.index.isin(features.index).all()
assert verified["is_deal"].isin([0, 1]).all(), "Некорректные метки в verified.csv"
assert features.columns.is_unique

feature_columns = features.columns.difference(["seller_id", "is_deal", "weight"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]

print(f"Объявлений в матрице: {len(X)}")
print(f"Признаков: {X.shape[1]}")
print(f"Объявлений с проверенными метками: {len(y)}")

### Дополнительные метки

Берём дополнительные метки с исходными весами 0.8 и 1.0, исключая все ID из `verified.csv`. Вес каждой дополнительной строки умножаем на общий коэффициент. Например, при коэффициенте 0.25 получаются веса 0.20 и 0.25. Вес проверенных меток равен 1.0. При нулевом коэффициенте обучаемся только на проверенных строках. Для повтора эксперимента без строк с весом 0.8 установите `INCLUDE_WEIGHT_08 = False`.

In [ ]:
extended = pd.read_csv(
    EXTENDED_VERIFIED_PATH,
    usecols=["listing_id", "is_deal", "weight"],
    index_col="listing_id",
)
assert extended.index.is_unique, "Повторяются ID в расширенной разметке"
assert extended.index.notna().all(), "В расширенной разметке есть пустые ID"

extra_labels = extended.loc[~extended.index.isin(verified.index)]
if not INCLUDE_WEIGHT_08:
    extra_labels = extra_labels.loc[extra_labels["weight"].ne(0.8)]
assert extra_labels.index.isin(X.index).all(), "Для дополнительных меток нет признаков"
assert extra_labels["is_deal"].isin([0, 1]).all(), "Некорректные дополнительные метки"
assert extra_labels["weight"].gt(0).all() and np.isfinite(extra_labels["weight"]).all()

print("Дополнительных меток:", len(extra_labels))
display(extra_labels.groupby(["is_deal", "weight"]).size())

## 2. Подготовить варианты модели

Берём ту же предобработку, что и в `models/model.ipynb`. На каждом обучении медианы, масштабирование и кодирование категорий рассчитываются по обучающим строкам. Для заблокированных объявлений ставим вероятность сделки 0.

In [ ]:
def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, make_column_selector(dtype_include=["number", "bool"])),
    ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=["number", "bool"])),
])
model = make_pipeline(
    preprocessor,
    LogisticRegression(C=1.0, max_iter=2000),
)

In [ ]:
def fit_model(model, X, y_train, extra_labels):
    train_labels = pd.concat([
        y_train.to_frame().assign(weight=1.0),
        extra_labels,
    ])
    X_fit = X.loc[train_labels.index]
    return clone(model).fit(
        X_fit,
        train_labels["is_deal"],
        logisticregression__sample_weight=train_labels["weight"],
    )

In [ ]:
blocked_ids = listings.index[listings["close_type"] == "заблокировано"]
X_verified = X.loc[y.index]

candidates = {}
for c in C_VALUES:
    for extra_weight in EXTRA_WEIGHT_SCALES:
        candidate = clone(model).set_params(logisticregression__C=c)
        labels = (
            extra_labels.iloc[:0] if extra_weight == 0
            else extra_labels.assign(weight=extra_labels["weight"] * extra_weight)
        )
        candidates[c, extra_weight] = (candidate, labels)

print("Комбинаций C и множителя:", len(candidates))
print("Порогов для каждой комбинации:", len(THRESHOLDS))

## 3. Подобрать порог внутри обучения

Делим проверенные строки обучающей части на четыре фолда. К каждому обучению добавляем выбранные дополнительные метки. Для каждого порога считаем F1 по всем внутренним отложенным прогнозам. При равном F1 выбираем порог ближе к 0,5.

In [ ]:
def deal_probabilities(fitted_model, X_test):
    probabilities = fitted_model.predict_proba(X_test)[:, 1]
    probabilities[X_test.index.isin(blocked_ids)] = 0
    return probabilities


def choose_threshold(candidate, y_train, labels):
    inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=INNER_SEED)
    probabilities = np.zeros(len(y_train))

    for train, test in inner_cv.split(y_train, y_train):
        fitted_model = fit_model(candidate, X, y_train.iloc[train], labels)
        X_test = X.loc[y_train.iloc[test].index]
        probabilities[test] = deal_probabilities(fitted_model, X_test)

    # Считаем F1 для всех порогов по одной матрице прогнозов.
    predictions = probabilities[:, None] > THRESHOLDS
    true_deals = y_train.to_numpy() == 1
    true_positives = (predictions & true_deals[:, None]).sum(axis=0)
    f1_scores = 2 * true_positives / (predictions.sum(axis=0) + true_deals.sum())

    best_indices = np.flatnonzero(f1_scores == f1_scores.max())
    best = best_indices[np.abs(THRESHOLDS[best_indices] - 0.5).argmin()]
    return THRESHOLDS[best], f1_scores[best]

## 4. Сравнить варианты на внешних фолдах

Для каждого из трёх разбиений собираем по одному внешнему прогнозу на каждое проверенное объявление. В таблице вариантов сравниваем F1 при фиксированных `C` и множителе, с порогом, выбранным внутри обучения. Отдельно оцениваем подбор всей конфигурации. Для этого на каждом внешнем фолде выбираем `C`, множитель весов и порог по внутреннему F1 до чтения меток внешнего фолда.

In [ ]:
def validation_metrics(predictions):
    shares = pd.DataFrame({"actual": y, "predicted": predictions})
    shares = shares.join(features["category"]).groupby("category")[["actual", "predicted"]].mean()
    return {
        "F1": f1_score(y, predictions),
        "Precision": precision_score(y, predictions, zero_division=0),
        "Recall": recall_score(y, predictions, zero_division=0),
        "CatShareMAE, п. п.": 100 * (shares["actual"] - shares["predicted"]).abs().mean(),
    }

In [ ]:
results = []
selection_results = []
selected_configs = []

for seed in CV_SEEDS:
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    predictions = pd.DataFrame(
        np.nan, index=y.index, columns=pd.MultiIndex.from_tuples(candidates),
    )
    selected_predictions = pd.Series(index=y.index, dtype=float)

    for fold, (train, test) in enumerate(outer_cv.split(X_verified, y), start=1):
        y_train = y.iloc[train]
        X_test = X_verified.iloc[test]
        inner_results = {}

        for (c, extra_weight), (candidate, labels) in candidates.items():
            threshold, inner_f1 = choose_threshold(candidate, y_train, labels)
            fitted_model = fit_model(candidate, X, y_train, labels)
            probabilities = deal_probabilities(fitted_model, X_test)
            predictions.loc[X_test.index, (c, extra_weight)] = (probabilities > threshold).astype(int)
            inner_results[c, extra_weight] = (inner_f1, threshold)

        best_config = max(inner_results, key=lambda config: inner_results[config][0])
        selected_predictions.loc[X_test.index] = predictions.loc[X_test.index, best_config]
        selected_configs.append({
            "Разбиение": seed, "Фолд": fold, "C": best_config[0], "Множитель": best_config[1],
            "Порог": inner_results[best_config][1],
        })

    assert predictions.notna().all().all() and selected_predictions.notna().all()
    for c, extra_weight in candidates:
        metrics = validation_metrics(predictions[c, extra_weight].astype(int))
        results.append({"Разбиение": seed, "C": c, "Множитель": extra_weight, **metrics})

    selection_results.append({
        "Разбиение": seed, **validation_metrics(selected_predictions.astype(int)),
    })
    print("Проверено разбиение:", seed, "| F1 подбора:", round(selection_results[-1]["F1"], 4))

## 5. Посмотреть лучший F1

Лидер таблицы выбран по среднему F1 на этих разбиениях. Его показатель может быть завышен из-за выбора среди нескольких вариантов. Средний F1 процедуры подбора считаем отдельно по прогнозам моделей, выбранных только на внутренних фолдах.

In [11]:
results = pd.DataFrame(results)
ranking = (
    results.groupby(["C", "Множитель"], as_index=False)
    .agg(
        F1=("F1", "mean"),
        F1_min=("F1", "min"),
        Precision=("Precision", "mean"),
        Recall=("Recall", "mean"),
        CatShareMAE=("CatShareMAE, п. п.", "mean"),
    )
    .sort_values(["F1", "CatShareMAE", "C", "Множитель"], ascending=[False, True, True, True])
    .reset_index(drop=True)
)

print("Лучшие настройки")
display(ranking.head(10).round(4))

print("Контрольная конфигурация при C=1 и множителе 0.5")
display(ranking.loc[ranking["C"].eq(1.0) & ranking["Множитель"].eq(0.5)].round(4))

print("Проверка процедуры подбора")
selection_results = pd.DataFrame(selection_results)
display(selection_results.round(4))
display(selection_results.drop(columns="Разбиение").mean().round(4))
display(pd.DataFrame(selected_configs).round(4))

Лучшие настройки
Контрольная конфигурация при C=1 и множителе 0.5
Проверка процедуры подбора


,C,Множитель,F1,F1_min,Precision,Recall,CatShareMAE
0,0.3,0.10,0.8177,0.8051,0.7925,0.8448,3.9853
1,0.1,0.25,0.8090,0.7983,0.7969,0.8218,4.6438
2,1.0,0.10,0.8081,0.7967,0.7722,0.8477,4.8394
3,3.0,0.10,0.8033,0.7901,0.7656,0.8448,4.9903
4,0.3,0.25,0.8023,0.7886,0.7687,0.8391,5.6720
5,0.1,0.10,0.8003,0.7826,0.7883,0.8132,4.8781
6,0.1,0.50,0.7995,0.7983,0.7759,0.8247,4.9348
7,1.0,0.25,0.7969,0.7871,0.7542,0.8448,5.7955
8,10.0,0.10,0.7925,0.7854,0.7377,0.8563,6.7810
9,3.0,0.25,0.7899,0.7854,0.7352,0.8534,6.8452


,C,Множитель,F1,F1_min,Precision,Recall,CatShareMAE
17,1.0,0.5,0.784,0.7773,0.7314,0.8448,6.6208


,Разбиение,F1,Precision,Recall,"CatShareMAE, п. п."
0,11,0.7869,0.7500,0.8276,5.2204
1,23,0.8205,0.8136,0.8276,3.8060
2,42,0.7552,0.7280,0.7845,4.6939


F1                    0.7875
Precision             0.7639
Recall                0.8132
CatShareMAE, п. п.    4.5734
dtype: float64

,Разбиение,Фолд,C,Множитель,Порог
0,11,1,1.0,0.10,0.63
1,11,2,0.3,0.25,0.65
2,11,3,0.3,0.25,0.59
3,11,4,1.0,0.25,0.67
4,11,5,0.1,0.00,0.35
5,23,1,0.1,0.10,0.59
6,23,2,0.1,0.00,0.46
7,23,3,0.1,0.00,0.44
8,23,4,1.0,0.10,0.65
9,23,5,0.1,0.00,0.28


## 6. Обучить модель с лучшими настройками

Берём лидера таблицы, подбираем один порог по четырём фолдам всей проверенной разметки и обучаем итоговую модель на доступных строках. Итоговые прогнозы остаются в таблице `tuned_predictions`.

In [12]:
best = ranking.iloc[0]
best_config = (best["C"], best["Множитель"])
candidate, labels = candidates[best_config]
threshold, _ = choose_threshold(candidate, y, labels)
final_model = fit_model(candidate, X, y, labels)

probabilities = deal_probabilities(final_model, X)
tuned_predictions = X.index.to_frame(index=False).assign(
    is_deal=(probabilities > threshold).astype(int),
)
assert tuned_predictions["listing_id"].is_unique
assert set(tuned_predictions["listing_id"]) == set(listings.index)
assert tuned_predictions.loc[tuned_predictions["listing_id"].isin(blocked_ids), "is_deal"].eq(0).all()

print("C:", best_config[0])
print("Множитель весов дополнительных меток:", best_config[1])
print("Итоговые веса дополнительных меток:", sorted(labels["weight"].round(3).unique().tolist()))
print("Итоговый порог:", round(threshold, 2))
print("Объявлений с прогнозом:", len(tuned_predictions))

C: 0.3
Множитель весов дополнительных меток: 0.1
Итоговые веса дополнительных меток: [0.08, 0.1]
Итоговый порог: 0.46
Объявлений с прогнозом: 15274
